# Q4 · System design: LLM extraction from ISW's daily Iran Updates

**Brief.** The PM sees the US–Iran war (which began on 28 Feb 2026) as a driver of global inflation and wants daily
updates scraped from ISW. An LLM extracts key facts from each article. The graded part is the **schema** and how
the **system design follows OOP principles**. Running it on the first articles after the war began is optional.

| File | Role |
|---|---|
| `schema.py` | Pydantic schema: controlled vocabulary (enums), event class hierarchy, `DailyBrief`, `BriefDiff` |
| `isw_pipeline.py` | The system: sources → parser → prompt → LLM client → extractor → repository → analyser |
| `examples.py` | Hand-labelled "gold" briefs for five real reports (28 Feb – 16 Mar 2026) |

**Contents:** 1. What the source looks like · 2. Design goals · 3. Architecture · 4. Schema · 5. OOP principles ·
6. Demo on the first weeks of the war · 7. Optional live LLM run · 8. Limitations

In [1]:
%load_ext autoreload
%autoreload 2
import json, logging, os, sys
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

logging.basicConfig(level=logging.WARNING, force=True)
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.width", 160, "display.max_colwidth", 120)

import schema as sc
import isw_pipeline as pp
from examples import GOLD
sys.path.insert(0, "../Q1_Q2"); import charts as ch          # same dark theme as the dashboard

---
## 1. What the source looks like

I read eight archived reports from 27 Feb (pre-war) to 16 Mar. The findings below drove the design.

| Observation | Design consequence |
|---|---|
| ISW writes about **military operations, not prices**. Price or market words appear 0–4 times in a ~4,000-word report | The LLM extracts **operational facts**. The schema maps them to **inflation channels**; prices come from our own data (Q1–Q3) |
| The Hormuz story moves through **stages**: warned vessels (28 Feb) → "may have started attacking" (1 Mar) → tanker hit, UKMTO incidents (2–4 Mar) → "has not attacked any vessels since March 12", safe passage (16 Mar) | An **ordered status ladder** for each chokepoint, so day-over-day change is computable |
| ISW grades its language: "likely" (56×), "reportedly" (56×), "claimed", "may have", "has not observed", "satellite imagery" | **Confidence, evidence type and source** are required fields, and the model must not upgrade them |
| Absence matters: "Iran has not attacked any vessels … since March 12" | A dedicated **negative-observation** event type |
| 2 Mar: Qatar halted all LNG output (≈20% of global supply) after drone attacks; Aramco shut Ras Tanura | **Named assets with an operator and output impact**: the facts that actually move inflation expectations |
| understandingwar.org and criticalthreats.org block scripts (Cloudflare, HTTP 403) | **Pluggable sources**: Internet Archive by default, direct or licensed feed or manual upload as alternatives |

---
## 2. Design goals
1. **Decision-useful.** Answer the PM's morning question: *what changed overnight that matters for inflation, and how
   sure are we?*
2. **Faithful.** Every fact carries a verbatim quote checked against the article, plus ISW's own confidence wording.
3. **Facts vs analytics.** The LLM extracts facts. Deterministic code turns them into channel scores and diffs, so
   the scoring is auditable, versioned and identical across LLMs.
4. **Swappable parts.** Source, LLM provider and storage can each change without touching the rest.

---
## 3. Architecture

```
            ┌──────────────────────── ExtractionPipeline (composition root) ─────────────────────────┐
            │                                                                                        │
 slugs ──▶  ArticleSource ──▶ ISWParser ──▶ BriefExtractor ───────────────▶ BriefRepository ──▶ BriefAnalyzer ──▶ PM
            (ABC)             HTML → text    │  PromptBuilder (from schema)  (ABC)               timeline, state,
             ├ InternetArchiveSource         │  LLMClient (ABC)              └ JsonFileRepository  diffs, Evaluator
             ├ DirectISWSource               │   ├ OpenRouterClient
             └ LocalFileSource               │   └ ReplayClient (tests)
                                             │  Pydantic validation ──✗──▶ error fed back to LLM (retry ≤3)
                                             └  verify_quotes() ──✗──▶ drop unverifiable events
```

| Component | Single responsibility |
|---|---|
| `ArticleSource` | Get raw HTML for a report (template method: cache → `_download()` → parse) |
| `ISWParser` | HTML → clean report text; date and report type from the URL |
| `PromptBuilder` | Build the system prompt **from the schema** (vocabulary, confidence ladder, one few-shot example) |
| `LLMClient` | One call: prompt + JSON schema → raw JSON text |
| `BriefExtractor` | JSON → validated `DailyBrief`; self-repair retries; hallucination guard |
| `BriefRepository` | Persist and reload briefs (one JSON file per report and model) |
| `BriefAnalyzer` / `Evaluator` | Timeline, running chokepoint state, diffs; scoring against gold labels |

---
## 4. The schema (`schema.py`)

**Controlled vocabulary (enums):** `InflationChannel` · `ChokepointStatus` (ordered ladder) · `Confidence` (ISW's
estimative ladder, ordered) · `EvidenceType` · `Chokepoint` · `Actor` · `AssetType` · `ReportType`.

**Event hierarchy.** One abstract base class and six concrete types, chosen by a `kind` discriminator:

```
ConflictEvent  (date, summary, initiator, location, confidence, sources[], first_occurrence, evidence_quote)
 ├─ Strike                 weapon, target_type, munitions_count
 ├─ MaritimeIncident       chokepoint, vessel, incident (warning / harassment / seizure / attack / mine …)
 ├─ InfrastructureAttack   asset (name, type, operator), damage, output_impact
 ├─ LeadershipTargeted     person, outcome
 ├─ Diplomacy              development (talks / ceasefire / sanctions …), mediator
 └─ NegativeObservation    activity, since          ("has not attacked … since March 12")
each implements channels(), direction(), severity() → weight() = severity × confidence
```

**Aggregate:** `DailyBrief` = report metadata + Key Takeaways (verbatim) + events + chokepoint assessments +
escalation trend + ISW's caveats. It provides `verify_quotes()`, the derived `channel_scores`, and
`diff(previous) → BriefDiff`.

The JSON schema is generated from these classes and sent to the LLM, and the field descriptions double as
instructions:

In [2]:
js = sc.DailyBrief.model_json_schema()
print(f"DailyBrief JSON schema: {len(json.dumps(js)):,} chars, {len(js['$defs'])} definitions")
print("Event types:", [d for d in js["$defs"] if d in ("Strike", "MaritimeIncident", "InfrastructureAttack",
                                                       "LeadershipTargeted", "Diplomacy", "NegativeObservation")])
print("\nSystem prompt built from the schema (first 1,200 chars):\n")
print(pp.PromptBuilder().system()[:1200])

DailyBrief JSON schema: 16,404 chars, 18 definitions
Event types: ['Diplomacy', 'InfrastructureAttack', 'LeadershipTargeted', 'MaritimeIncident', 'NegativeObservation', 'Strike']

System prompt built from the schema (first 1,200 chars):

You extract facts from ISW-CTP 'Iran Update' reports for an inflation-focused portfolio manager.
Return ONE JSON object matching the DailyBrief schema. Rules:
1. Extract only what the report states. Never infer prices or add outside knowledge.
2. Every evidence_quote must be copied verbatim from the report (no paraphrase, max 400 chars).
3. Map confidence from ISW's own wording; never upgrade it:
- CONFIRMED: confirmed, satellite imagery, footage, geolocated
- ALMOST_CERTAIN: almost certainly
- LIKELY: likely, ISW-CTP assesses, probably
- POSSIBLE: may, may have, could, appears to, suggests
- CLAIMED: reportedly, claimed, according to
- UNVERIFIED: unconfirmed, unverified, cannot independently verify
- NOT_OBSERVED: has not observed, has not attacked, 

---
## 5. How the design follows OOP principles

| Principle | Where | Why it helps here |
|---|---|---|
| **Abstraction** | `ArticleSource`, `LLMClient`, `BriefRepository` are abstract base classes; `ConflictEvent` is the abstract event | The pipeline is written against *what* a component does, not *how*. Switching ISW access route or LLM vendor is a one-line change |
| **Encapsulation** | Rules live inside the models: CONFIRMED needs hard evidence; one assessment per chokepoint; immutable (`frozen`) objects; `verify_quotes()` on the brief | Invalid states can't be built. An LLM answer that breaks a rule fails validation and is sent back for repair |
| **Inheritance** | Event subclasses share `ConflictEvent`'s fields and validator; sources share `ArticleSource.get()` (template method) | Common behaviour is written once; a subclass adds only what's specific to it |
| **Polymorphism** | `channels()`, `direction()`, `severity()` differ per event type; `DailyBrief.channel_scores` just loops over events. The `kind` discriminator picks the class when parsing JSON | Adding an event type doesn't change scoring or diffing code |
| **Composition** | `ExtractionPipeline` = source + extractor + repository; `BriefExtractor` = LLM client + prompt builder; `DailyBrief` aggregates events | Parts are assembled, not hard-wired, so each is testable on its own |
| **S**ingle responsibility | Parser parses, extractor validates, analyser analyses, repository stores | Changes stay local (e.g. ISW redesigns its HTML → only `ISWParser` changes) |
| **O**pen/closed | New `ArticleSource` or event type = new subclass; no edits to existing classes | Safe to extend under time pressure |
| **L**iskov substitution | Any `ArticleSource` / `LLMClient` works in the pipeline: `ReplayClient` replaces `OpenRouterClient` in tests | Deterministic, offline tests of the full pipeline |
| **I**nterface segregation | Each interface has one or two methods (`_download`, `complete`, `save` / `load_all`) | Implementations stay trivial |
| **D**ependency inversion | `BriefExtractor` depends on the `LLMClient` abstraction, injected by the caller | No vendor lock-in; cost or model changes don't touch extraction logic |

---
## 6. Demo: the first weeks of the war

The full pipeline runs on five real reports from the Internet Archive. The LLM is played by `ReplayClient`,
which returns the hand-labelled gold briefs, so the run is deterministic and needs no API key. Two faults are
injected to show the safeguards working:
* **28 Feb (special):** the first answer claims "confirmed" on a social-media source. Validation fails, the error is
  fed back, and the second answer passes.
* **2 Mar:** one extra event carries a quote that isn't in the article. `verify_quotes()` drops it.

In [3]:
slugs = list(GOLD)
answers = {s: b.model_dump_json() for s, b in GOLD.items()}

# fault 1: invalid first answer for 28 Feb (CONFIRMED on a social-media source) -> self-repair retry
bad = json.loads(answers[slugs[0]])
bad["events"][0]["confidence"] = 6
bad["events"][0]["sources"] = [{"outlet": "X/Twitter OSINT account", "evidence": "social_media"}]
class FlakyReplay(pp.ReplayClient):
    def __init__(self, answers, first_bad):
        super().__init__(answers); self.first_bad, self.used = first_bad, set()
    def complete(self, system, user, schema):
        slug = user.split("SLUG: ")[1].split()[0]
        if slug in self.first_bad and slug not in self.used:
            self.used.add(slug); return json.dumps(self.first_bad[slug])
        return super().complete(system, user, schema)

# fault 2: a hallucinated event for 2 Mar (quote not in the article) -> dropped
mar2 = json.loads(answers[slugs[2]])
mar2["events"].append({"kind": "infrastructure_attack", "summary": "Kharg Island export terminal destroyed.",
                       "initiator": "israel", "confidence": 4, "sources": [], "damage": "destroyed",
                       "asset": {"name": "Kharg Island", "asset_type": "export_terminal"},
                       "evidence_quote": "Israel destroyed the Kharg Island oil export terminal on March 2."})
answers[slugs[2]] = json.dumps(mar2)

llm = FlakyReplay(answers, first_bad={slugs[0]: bad})
pipe = pp.ExtractionPipeline(source=pp.InternetArchiveSource(),
                             extractor=pp.BriefExtractor(llm, pp.PromptBuilder(few_shot=[GOLD[slugs[1]]])),
                             repo=pp.JsonFileRepository())
results = pipe.run(slugs)
display(pd.DataFrame([{"report": f"{r.article.report_date:%d %b} {r.article.report_type.value}",
                       "words in article": len(r.article.text.split()), "attempts": r.attempts,
                       "validation errors fed back": r.errors[0][:90] + "…" if r.errors else "",
                       "events kept": len(r.brief.events) if r.brief else 0,
                       "dropped (quote not in article)": "; ".join(r.dropped_events)} for r in results]).set_index("report"))

,words in article,attempts,validation errors fed back,events kept,dropped (quote not in article)
report,,,,,
28 Feb special,1512,2,"events.0.strike: Value error, CONFIRMED requires imagery, footage, geolocation or an offic…",2,
28 Feb evening_special,4166,1,,1,
02 Mar evening_special,5170,1,,3,Kharg Island export terminal destroyed.
04 Mar evening_special,5560,1,,1,
16 Mar evening_special,3603,1,,2,


In [4]:
an = pp.BriefAnalyzer([r.brief for r in results if r.brief])
tl = an.timeline()
state = an.chokepoint_state()
display(pd.concat([state.rename("Hormuz status (carried forward)"), tl], axis=1))

# chart: flow (channel scores per report) and stock (Hormuz status ladder)
chan_cols = [c for c in tl.columns if c not in ("events", "trend")]
palette = ["#d95926", "#3987e5", "#199e70", "#c98500", "#9085e9", "#e66767", "#d55181", "#8a94a6"]
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.62, 0.38], vertical_spacing=0.1,
                    subplot_titles=("Confidence-weighted signal by inflation channel (news in each report)",
                                    "Strait of Hormuz status (carried forward)"))
for c, col in zip(chan_cols, palette):
    fig.add_trace(go.Bar(x=tl.index, y=tl[c], name=c.replace("_", " "), marker_color=col,
                         hovertemplate="%{y:+.2f}<extra>" + c + "</extra>"), 1, 1)
lvl = state.map(lambda s: sc.ChokepointStatus[s].value)
fig.add_trace(go.Scatter(x=state.index, y=lvl, mode="lines+markers+text", text=state.values, textposition="top center",
                         line=dict(color="#e6e9ef", shape="hv"), showlegend=False,
                         textfont=dict(size=10, color="#c9d1dc")), 2, 1)
lay = ch.base_layout("", 620); lay.pop("xaxis"); lay.pop("yaxis"); lay["barmode"] = "relative"
lay["legend"].update(y=-0.12, yanchor="top", x=0.5, xanchor="center"); lay["hovermode"] = "x unified"
fig.update_layout(**lay); fig.update_xaxes(gridcolor=ch.GRID_COLOR); fig.update_yaxes(gridcolor=ch.GRID_COLOR)
fig.update_yaxes(range=[-0.5, 7], tickvals=[s.value for s in sc.ChokepointStatus],
                 ticktext=[s.name.replace("_", " ").lower() for s in sc.ChokepointStatus], row=2, col=1)
fig.show()

,Hormuz status (carried forward),events,trend,escalation_intensity,geographic_spread,regime_stability,shipping_chokepoint,energy_supply,de_escalation
28 Feb special,OPEN,2,escalating,1.33,1.33,1.33,0.00,0.00,0.0
28 Feb evening special,THREATENED,1,escalating,0.00,0.00,0.00,0.67,0.00,0.0
02 Mar evening special,CONFIRMED_ATTACKS,3,escalating,0.00,7.67,0.00,1.67,7.67,0.0
04 Mar evening special,CONFIRMED_ATTACKS,1,stable,0.00,0.00,0.00,3.33,0.00,0.0
16 Mar evening special,MANAGED_PASSAGE,2,de_escalating,0.00,0.00,0.00,0.00,0.00,-1.5


In [5]:
rows = []
for d in an.diffs():
    rows.append({"from → to": f"{d.from_date:%d %b} → {d.to_date:%d %b}",
                 "chokepoint change": ", ".join(f"{k.value}: {a.name} → {b.name}" for k, (a, b) in d.chokepoint_changes.items()) or "—",
                 "biggest channel moves": ", ".join(f"{k} {v:+.1f}" for k, v in sorted(d.channel_delta.items(), key=lambda kv: -abs(kv[1]))[:3] if v),
                 "new 'first' events (only if the article says so)": "; ".join(d.new_firsts) or "—"})
display(pd.DataFrame(rows).set_index("from → to"))

,chokepoint change,biggest channel moves,new 'first' events (only if the article says so)
from → to,,,
28 Feb → 28 Feb,strait_of_hormuz: OPEN → THREATENED,"regime_stability -1.3, geographic_spread -1.3, escalation_intensity -1.3",—
28 Feb → 02 Mar,strait_of_hormuz: THREATENED → CONFIRMED_ATTACKS,"geographic_spread +7.7, energy_supply +7.7, shipping_chokepoint +1.0",—
02 Mar → 04 Mar,—,"energy_supply -7.7, geographic_spread -7.7, shipping_chokepoint +1.7",—
04 Mar → 16 Mar,strait_of_hormuz: CONFIRMED_ATTACKS → MANAGED_PASSAGE,"shipping_chokepoint -3.3, de_escalation -1.5",—


**How the extracted information changed at the start of the conflict**
* **28 Feb (morning):** the strike campaign and Iran's retaliation against **US bases in six regional states**
  (Bahrain, Qatar, UAE, Kuwait, Saudi Arabia, Jordan) dominate. The signal is escalation and geographic spread, plus
  regime-stability risk from decapitation strikes.
* **28 Feb (evening):** Hormuz enters the record as **threatened**. The IRGC warns ships ("no ship is allowed to
  pass"), but ISW has *not* observed attacks. Shipping risk is flagged, at low confidence.
* **2 Mar:** the inflation-relevant step change. **Qatar halts all LNG production** (≈20% of global supply) after
  drone attacks on Ras Laffan, Aramco shuts **Ras Tanura**, and a tanker is reportedly hit in Bahrain. Hormuz moves to
  **confirmed attacks**, and the energy-supply and Gulf-spread channels jump.
* **4 Mar:** fewer new energy facts, but shipping risk stays high: UKMTO reports five incidents, "likely Iranian
  attacks". The status is unchanged, so the *stock* of risk is still high even though the *flow* of news fell.
* **16 Mar:** "has not attacked any vessels … since March 12" and **safe passage** for some oil tankers. Hormuz
  steps down to **managed passage** and the signal turns to de-escalation.

That sequence (threat → energy-infrastructure shock → sustained shipping risk → managed passage) is what you'd want
to line up against oil and LNG prices and against Truflation's energy component from Q3, which rose about 14% YoY.

**Reading the scores:** channel scores measure *news flow* in each report. The chokepoint status carries forward
and measures the *state*. Both are needed: a quiet day isn't a safe day.

---
## 7. Optional: live LLM run (OpenRouter free models)

The same pipeline with `OpenRouterClient` instead of the replay client, scored against the gold labels. It runs
only if `OPENROUTER_API_KEY` is set (a free key from openrouter.ai). Not graded.

In [6]:
if os.getenv("OPENROUTER_API_KEY"):
    llm = pp.OpenRouterClient(model=os.getenv("OPENROUTER_MODEL", "meta-llama/llama-3.3-70b-instruct:free"))
    live = pp.ExtractionPipeline(pp.InternetArchiveSource(), pp.BriefExtractor(llm, pp.PromptBuilder(few_shot=[GOLD[slugs[1]]])),
                                 pp.JsonFileRepository()).run(slugs)
    display(pd.DataFrame({f"{r.article.report_date:%d %b} {r.article.report_type.value}":
                          (pp.Evaluator.score(r.brief, GOLD[r.article.slug]) if r.brief else {"failed": r.errors[-1][:120]})
                          for r in live}).T)
else:
    print("OPENROUTER_API_KEY not set - skipping the live run. The replay demo above exercises the same code path.")

OPENROUTER_API_KEY not set - skipping the live run. The replay demo above exercises the same code path.


---
## 8. Limitations and next steps
* **Access.** ISW blocks scripted access; the Internet Archive lags by days and doesn't have every report. For
  production I'd ask ISW for a feed or licence, or ingest their email briefing.
* **LLM variance.** Free models differ in how well they follow a schema. The design limits the damage (schema
  validation, self-repair retries, verbatim-quote checks, temperature 0), and `Evaluator` measures it against gold
  labels. A larger gold set (20–30 reports) would be the next step.
* **Scoring is a heuristic.** Severity and weights are transparent rules, not a calibrated model. The next step is
  to regress Q3's energy-component surprises and oil/LNG price moves on these channel scores to calibrate them.
* **Long reports.** Reports run to ~5,500 words, within context limits for current models. For longer inputs, split
  by section and merge the briefs, which works because events are independent and chokepoints are keyed.